# Grid Finding — Orchestrator

Runs the Grid Finding pipeline **per borough**: for each borough in the list, notebooks 01–06 extract features, merge into a combined CSV, then run ML (07) and heatmap (08). Each borough gets its own timestamped folder. After all boroughs finish, notebook 09 generates a cross-borough comparison (if 2+ boroughs).

**Pipeline (per borough):**
```
01_grid_definition          → cell_id, zone_type (Y variable)
02_amenity_composition      → amenity_density, amenity_ratio_food_drink (OSM)
03_building_characteristics → avg_floors, avg_yearbuilt, building_count, total_bldg_area (PLUTO)
04_land_use_mix             → landuse_entropy (PLUTO)
05_tourism_intensity        → tourism_density (OSM)
06_commercial_density       → shop_density_km2, shop_type_entropy, brand_ratio (OSM)
──────────────────────────────────────────────────────────────────
07_ml_classification        → train & evaluate models, export predictions
08_heatmap_visualization    → static + interactive heatmaps
```
**After all boroughs:**
```
09_comparison               → cross-borough comparison (auto-skip if < 2 boroughs)
```

**Usage:** Change parameters in the cell below → Restart kernel → Run All Cells

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║           PIPELINE PARAMETERS — CHANGE HERE         ║
# ╚══════════════════════════════════════════════════════╝

# ── Borough selection ─────────────────────────────────
# Available boroughs in NYC PLUTO:
#   "MN"  →  Manhattan      (~1,810 cells at 150m)
#   "BK"  →  Brooklyn       (~4,500+ cells at 150m)
#   "QN"  →  Queens         (~6,000+ cells at 150m)
#   "BX"  →  Bronx          (~2,500+ cells at 150m)
#   "SI"  →  Staten Island  (~2,000+ cells at 150m)
#
# You can combine boroughs: ["MN", "BK"] runs both.
# Each borough runs independently with its own folder.
# Cell counts are estimates — actual count depends on
# lot density and min_lots_per_cell threshold.

BOROUGH = ["MN"]            # ← CHANGE THIS

# ── Grid parameters ──────────────────────────────────
CELL_SIZE_M = 150            # grid cell size in meters
MIN_LOTS_PER_CELL = 3        # cells with fewer lots are dropped

# ── Classification mode ──────────────────────────────
INCLUDE_OTHER = False         # False = binary (Commercial vs Residential)
                              # True  = 3-class (+ Other)

# ── PLUTO path ───────────────────────────────────────
PLUTO_PATH = "../ramy/NYC_pluto_25v4_csv/pluto_25v4.csv"

In [ ]:
# ── Validate parameters + shared constants ────────────
import json
from datetime import datetime

_BOROUGH_CODES = {"MN": 1, "BX": 2, "BK": 3, "QN": 4, "SI": 5}
_BOROUGH_NAMES = {"MN": "Manhattan", "BK": "Brooklyn", "QN": "Queens",
                  "BX": "Bronx", "SI": "Staten Island"}

for b in BOROUGH:
    if b not in _BOROUGH_CODES:
        raise ValueError(f"Unknown borough '{b}'. Use: {list(_BOROUGH_CODES.keys())}")

_TIMESTAMP = datetime.now().strftime("%Y-%m-%d_%Hh%M")

_ZONE_TYPE_RULES = {
    "Residential": {"landuse": "01", "threshold": 0.70},
    "Commercial": {"landuse": "02", "threshold": 0.50},
    "Industrial": {"landuse": "05", "threshold": 0.30},
    "Institutional": {"landuse": "08", "threshold": 0.30},
    "Open Space": {"landuse": ["09", "11"], "threshold": 0.30},
    "Mixed-Use": {"description": "No single category dominates"},
}

borough_names = ", ".join(_BOROUGH_NAMES[b] for b in BOROUGH)
print(f"Boroughs to run: {borough_names} ({BOROUGH})")
print(f"Timestamp:       {_TIMESTAMP}")
print(f"Cell size:       {CELL_SIZE_M}m")
print(f"Include Other:   {INCLUDE_OTHER}")

In [ ]:
import papermill as pm
import pandas as pd
import pathlib
import time
import os
import re
import tempfile

PIPELINE = [
    ("01_grid_definition.ipynb",          "01 · Grid Definition"),
    ("02_amenity_composition.ipynb",      "02 · Amenity Composition"),
    ("03_building_characteristics.ipynb", "03 · Building Characteristics"),
    ("04_land_use_mix.ipynb",             "04 · Land Use Mix"),
    ("05_tourism_intensity.ipynb",        "05 · Tourism Intensity"),
    ("06_commercial_density.ipynb",       "06 · Commercial Density"),
]

KERNEL_NAME = "python3"

print(f"papermill {pm.__version__}")
print(f"Pipeline: {len(PIPELINE)} feature notebooks + ML + heatmap per borough")

In [ ]:
# ── Helper: run a notebook via papermill ──────────────

def run_notebook(nb_path, description, params=None):
    """Run a notebook, return (status, elapsed)."""
    if not pathlib.Path(nb_path).exists():
        raise FileNotFoundError(f"Notebook not found: {nb_path}")
    tmp = pathlib.Path(tempfile.mktemp(suffix=".ipynb"))
    t0 = time.time()
    try:
        pm.execute_notebook(
            nb_path, str(tmp),
            kernel_name=KERNEL_NAME,
            parameters=params or {},
            progress_bar=False,
        )
        elapsed = time.time() - t0
        print(f"  OK  ({elapsed:.1f} s)")
        return "ok", elapsed
    except pm.exceptions.PapermillExecutionError as e:
        elapsed = time.time() - t0
        print(f"  FAILED  ({elapsed:.1f} s)")
        print(f"  Error: {e}")
        return "failed", elapsed
    finally:
        if tmp.exists():
            tmp.unlink()

In [ ]:
# ══════════════════════════════════════════════════════
#  MAIN LOOP — run full pipeline per borough
# ══════════════════════════════════════════════════════

all_borough_results = {}

for borough_code in BOROUGH:
    borough_name = _BOROUGH_NAMES[borough_code]
    folder_name = f"{borough_name}_{_TIMESTAMP}"
    csv_dir = f"csv/{folder_name}"
    plots_dir = f"outputs/{folder_name}"

    os.makedirs(csv_dir, exist_ok=True)
    os.makedirs(plots_dir, exist_ok=True)

    print(f"\n{'#'*60}")
    print(f"  BOROUGH: {borough_name} ({borough_code})")
    print(f"  CSV:     {csv_dir}/")
    print(f"  Plots:   {plots_dir}/")
    print(f"{'#'*60}")

    # ── Write grid.json for this borough ──────────────
    config = {
        "borough_filter": [borough_code],
        "borough_codes": _BOROUGH_CODES,
        "grid_cell_size_m": CELL_SIZE_M,
        "min_lots_per_cell": MIN_LOTS_PER_CELL,
        "pluto_path": PLUTO_PATH,
        "include_other": INCLUDE_OTHER,
        "csv_dir": csv_dir,
        "feature_flags": {"needs_pluto": True},
        "zone_type_rules": _ZONE_TYPE_RULES,
    }
    with open("grid.json", "w", encoding="utf-8") as f:
        json.dump(config, f, indent=4)

    # ── Run feature extraction notebooks (01–06) ─────
    results = []
    for nb_path, description in PIPELINE:
        print(f"\n  {description}")
        status, elapsed = run_notebook(
            nb_path, description,
            params={"GRID_CONFIG": "grid.json"},
        )
        results.append((nb_path, status, elapsed))

    failed = [nb for nb, s, _ in results if s == "failed"]
    if failed:
        print(f"\n  WARNING: {len(failed)} notebook(s) failed for {borough_name}: {failed}")

    # ── Merge CSVs ───────────────────────────────────
    csv_files = [
        f"{csv_dir}/01_grid_definition.csv",
        f"{csv_dir}/02_amenity_composition.csv",
        f"{csv_dir}/03_building_characteristics.csv",
        f"{csv_dir}/04_land_use_mix.csv",
        f"{csv_dir}/05_tourism_intensity.csv",
        f"{csv_dir}/06_commercial_density.csv",
    ]

    df_combined = pd.read_csv(csv_files[0], dtype={"cell_id": str})
    for csv_path in csv_files[1:]:
        if pathlib.Path(csv_path).exists():
            df_other = pd.read_csv(csv_path, dtype={"cell_id": str})
            merge_cols = [c for c in df_other.columns if c != "cell_id"]
            df_combined = df_combined.merge(
                df_other[["cell_id"] + merge_cols],
                on="cell_id", how="left",
            )

    combined_path = f"{csv_dir}/combined_grid.csv"
    df_combined.to_csv(combined_path, index=False, encoding="utf-8")
    print(f"\n  Combined: {df_combined.shape[0]} cells x {df_combined.shape[1]} cols")

    # ── Run ML (07) ──────────────────────────────────
    print(f"\n  07 · ML Classification")
    run_notebook(
        "07_ml_classification.ipynb", "ML",
        params={"CSV_PATH": combined_path, "PLOTS_DIR": plots_dir},
    )

    # ── Run Heatmap (08) ─────────────────────────────
    print(f"\n  08 · Heatmap Visualization")
    run_notebook(
        "08_heatmap_visualization.ipynb", "Heatmap",
        params={"PLOTS_DIR": plots_dir},
    )

    all_borough_results[borough_name] = {
        "cells": len(df_combined),
        "csv_dir": csv_dir,
        "plots_dir": plots_dir,
        "results": results,
    }

    print(f"\n  {borough_name} DONE — {len(df_combined)} cells")

# ── Borough summary ──────────────────────────────────
print(f"\n{'#'*60}")
print(f"  ALL BOROUGHS COMPLETE")
print(f"{'#'*60}")
for name, info in all_borough_results.items():
    print(f"  {name:<20s} {info['cells']:>6d} cells  →  {info['plots_dir']}/")
print()

In [ ]:
# ── Run comparison notebook (if 2+ unique boroughs) ──

_ts_re = re.compile(r"^(.+)_(\d{4}-\d{2}-\d{2}_\d{2}h\d{2})$")
_unique_boroughs = set()
for d in pathlib.Path("csv").iterdir():
    if d.is_dir() and (d / "07_predictions.csv").exists():
        m = _ts_re.match(d.name)
        _unique_boroughs.add(m.group(1) if m else d.name)

if len(_unique_boroughs) >= 2:
    print(f"{'='*60}")
    print(f"  09 · Borough Comparison ({len(_unique_boroughs)} boroughs)")
    print(f"{'='*60}")
    run_notebook(
        "09_comparison.ipynb", "Comparison",
        params={"PLOTS_DIR": "outputs/Comparison"},
    )
else:
    print(f"SKIP  09 · Borough Comparison (need 2+ boroughs, found {len(_unique_boroughs)})")

print(f"\n{'#'*60}")
print(f"  GRID FINDING PIPELINE COMPLETE")
print(f"{'#'*60}")
for name, info in all_borough_results.items():
    print(f"  {name}: {info['plots_dir']}/")
if len(_unique_boroughs) >= 2:
    print(f"  Comparison: outputs/Comparison/")